In [ ]:
!pip install transformers datasets==2.19.1 evaluate seqeval -q

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "en")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

In [ ]:
print("Sample Tokens:", dataset["train"][0]["tokens"])
print("Sample Labels:", dataset["train"][0]["ner_tags"])

Sample Tokens: ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')']
Sample Labels: [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0]


In [ ]:

# Reduce dataset size for faster training
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_val = dataset["validation"].shuffle(seed=42).select(range(500))

dataset["train"] = small_train
dataset["validation"] = small_val

In [ ]:
# Load tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
zed_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
zed_dataset.set_format("torch")

In [ ]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 500
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 2000
    })
})
{'tokens': ["''", 'January', '21', "''", '–', 'Nanny', 'and', 'the', 'Professor'], 'ner_tags': [0, 0, 0, 0, 0, 1, 2, 2, 2], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['PER: Nanny and the Professor']}


In [ ]:

# Get label names from dataset
label_list = dataset["train"].features["ner_tags"].feature.names

print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [ ]:

from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
model.to(device)

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [ ]:

from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]

    true_preds = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    max_steps=375, # 3 epochs * (2000 samples / 16 batch size) = 375 steps
    weight_decay=0.01,
    logging_steps=50,
    fp16=True # Enable mixed precision training for potential speedup on compatible GPUs
)

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=zed_dataset["train"],
    eval_dataset=zed_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

Step,Training Loss
50,1.348716
100,0.731959
150,0.497631
200,0.428202
250,0.343696
300,0.282544
350,0.323045


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.5481069869995118, metrics={'train_runtime': 78.1729, 'train_samples_per_second': 76.753, 'train_steps_per_second': 4.797, 'total_flos': 783989471232000.0, 'train_loss': 0.5481069869995118, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()
print(results)


{'eval_loss': 0.34491100907325745, 'eval_precision': 0.6589861751152074, 'eval_recall': 0.7361647361647362, 'eval_f1': 0.6954407294832827, 'eval_accuracy': 0.8957911321179384, 'eval_runtime': 2.0743, 'eval_samples_per_second': 241.047, 'eval_steps_per_second': 15.427, 'epoch': 3.0}


In [ ]:
sentence = "John works at Google in California"

# Tokenize properly
inputs = tokenizer(sentence, return_tensors="pt")

# Move inputs to same device as model
inputs = {k: v.to(device) for k, v in inputs.items()}

# Get predictions
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)

# Convert tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Map labels
predicted_labels = [label_list[p.item()] for p in predictions[0]]

# Print output (ignore special tokens)
for token, label in zip(tokens, predicted_labels):
    if token not in ["[CLS]", "[SEP]"]:
        print(f"{token}: {label}")

john: B-PER
works: I-PER
at: O
google: B-ORG
in: I-ORG
california: I-ORG
